# Chicago Affordable Housing Data Cleaning

## Purpose

This notebook cleans Chicago's Affordable Rental Housing Developments dataset for later geographic and resource-allocation analysis.

The cleaning process standardizes text fields, converts numeric columns, removes system-generated fields, checks geographic values, documents missing data and saves a reproducible processed dataset.

## Important interpretation

This dataset represents documented City-supported affordable developments. It does not represent currently vacant units, waiting-list availability or housing that is immediately accessible to someone experiencing homelessness.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

print("Project root:", project_root)

Project root: /workspace/scratch/5b8a8dcae770/chicago-homelessness-resource-allocation


In [3]:
raw_path = (
    project_root
    / "data"
    / "raw"
    / "affordable_rental_housing_developments.csv"
)

affordable_housing = pd.read_csv(raw_path)

print("Raw dataset dimensions:", affordable_housing.shape)
display(affordable_housing.head())

Raw dataset dimensions: (598, 11)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,latitude,longitude
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,41.932073,-87.712872
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,41.869878,-87.624269
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,41.917643,-87.692170
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,41.845653,-87.617816
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,41.861881,-87.657558


In [4]:
print("Columns in the raw dataset:")

for column in affordable_housing.columns:
    print(column)

Columns in the raw dataset:
community_area
community_area_number
property_type
property_name
address
zip_code
phone_number
management_company
units
latitude
longitude


In [5]:
selected_columns = [
    "community_area",
    "community_area_number",
    "property_type",
    "property_name",
    "address",
    "zip_code",
    "phone_number",
    "management_company",
    "units",
    "latitude",
    "longitude",
]

missing_columns = [
    column
    for column in selected_columns
    if column not in affordable_housing.columns
]

assert not missing_columns, f"Required columns are missing: {missing_columns}"

housing_clean = affordable_housing[selected_columns].copy()

print("Selected dataset dimensions:", housing_clean.shape)
display(housing_clean.head())

Selected dataset dimensions: (598, 11)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,latitude,longitude
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,41.932073,-87.712872
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,41.869878,-87.624269
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,41.917643,-87.692170
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,41.845653,-87.617816
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,41.861881,-87.657558


In [6]:
numeric_columns = [
    "community_area_number",
    "units",
    "latitude",
    "longitude",
]

for column in numeric_columns:
    housing_clean[column] = pd.to_numeric(
        housing_clean[column],
        errors="coerce",
    )

housing_clean["community_area_number"] = (
    housing_clean["community_area_number"].astype("Int64")
)

housing_clean["units"] = housing_clean["units"].astype("Int64")

display(housing_clean[numeric_columns].head())

,community_area_number,units,latitude,longitude
0,21,25,41.932073,-87.712872
1,32,23,41.869878,-87.624269
2,22,1,41.917643,-87.692170
3,35,134,41.845653,-87.617816
4,28,7,41.861881,-87.657558


In [7]:
# Normalize whitespace without replacing missing values with strings.
text_columns = housing_clean.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    housing_clean[column] = (
        housing_clean[column].astype("string").str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Apply only reviewed source corrections. Keep the review trail in docs.
correction_path = project_root / "docs" / "HOUSING_CATEGORY_CORRECTIONS.csv"
corrections = pd.read_csv(correction_path)
for correction in corrections.itertuples(index=False):
    match = (
        housing_clean["property_name"].eq(correction.property_name)
        & housing_clean["address"].eq(correction.address)
    )
    assert match.sum() == 1, f"Review correction match: {correction.property_name}"
    housing_clean.loc[match, "community_area"] = correction.community_area
    housing_clean.loc[match, "community_area_number"] = correction.community_area_number

# Merge spelling errors only. Keep distinct housing programs separate.
housing_clean["property_type"] = housing_clean["property_type"].replace({
    "Mutifamily": "Multifamily",
    "Multfamily": "Multifamily",
})

assert housing_clean.groupby("community_area_number")["community_area"].nunique().le(1).all()
assert housing_clean.groupby("community_area")["community_area_number"].nunique().le(1).all()
print("Reviewed category corrections applied.")


Reviewed category corrections applied.


In [8]:
housing_clean["units_missing"] = housing_clean["units"].isna()

print("Rows with missing unit counts:")
display(
    housing_clean.loc[
        housing_clean["units_missing"],
        [
            "property_name",
            "address",
            "community_area",
            "units",
            "units_missing",
        ],
    ]
)

Rows with missing unit counts:


,property_name,address,community_area,units,units_missing
27,Arthur on Aberdeen,210 N. Aberdeen St.,Near West Side,<NA>,True


In [9]:
invalid_community_areas = housing_clean.loc[
    housing_clean["community_area_number"].notna()
    & ~housing_clean["community_area_number"].between(1, 77)
]

invalid_latitudes = housing_clean.loc[
    housing_clean["latitude"].notna()
    & ~housing_clean["latitude"].between(41.6, 42.1)
]

invalid_longitudes = housing_clean.loc[
    housing_clean["longitude"].notna()
    & ~housing_clean["longitude"].between(-88.0, -87.5)
]

print("Invalid community-area numbers:", len(invalid_community_areas))
print("Suspicious latitude values:", len(invalid_latitudes))
print("Suspicious longitude values:", len(invalid_longitudes))

Invalid community-area numbers: 0
Suspicious latitude values: 0
Suspicious longitude values: 0


In [10]:
duplicate_count = housing_clean.duplicated().sum()

missing_summary = (
    housing_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percent"] = (
    housing_clean.isna().mean() * 100
).round(1)

print("Exact duplicate rows:", duplicate_count)
display(missing_summary.sort_values("missing_percent", ascending=False))

Exact duplicate rows: 0


,missing_count,missing_percent
units,1,0.2
community_area,0,0.0
property_type,0,0.0
community_area_number,0,0.0
property_name,0,0.0
address,0,0.0
phone_number,0,0.0
zip_code,0,0.0
management_company,0,0.0
latitude,0,0.0


In [11]:
assert not housing_clean.empty, "The cleaned dataset is empty."
assert housing_clean.columns.is_unique, "Duplicate column names were found."
assert housing_clean["community_area_number"].dropna().between(1, 77).all()
assert housing_clean["units"].dropna().gt(0).all()

print("All cleaning validation checks passed.")

All cleaning validation checks passed.


In [12]:
clean_path = (
    project_root
    / "data"
    / "processed"
    / "affordable_housing_cleaned.csv"
)

housing_clean.to_csv(clean_path, index=False)

saved_clean = pd.read_csv(clean_path)

assert saved_clean.shape == housing_clean.shape

print("Cleaned dataset saved to:", clean_path)
print("Saved dataset dimensions:", saved_clean.shape)
display(saved_clean.head())

Cleaned dataset saved to: /workspace/scratch/5b8a8dcae770/chicago-homelessness-resource-allocation/data/processed/affordable_housing_cleaned.csv
Saved dataset dimensions: (598, 12)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,latitude,longitude,units_missing
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,41.932073,-87.712872,False
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,41.869878,-87.624269,False
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,41.917643,-87.692170,False
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,41.845653,-87.617816,False
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,41.861881,-87.657558,False
